Extract values at the surface (pressure and geopotential) for plotting

In [30]:
import numpy as np
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import matplotlib
import matplotlib.colors as colors

In [31]:
def z_interp(h, field_vals_all, lon, lat, z_val):
    field_vals = np.zeros((len(lat), len(lon)))
    for i in np.arange(len(lat)):
        for j in np.arange(len(lon)):
            if h[-1,i,j] > z_val:
                # This value is inside the topography
                field_vals[i,j] = np.nan
            else:
                # Find indices either side of this value
                low_idx = np.where(h[:,i,j] < z_val)[0][0]
                high_idx = np.where(h[:,i,j] > z_val)[0][-1]

                # Compute weightings
                weight_low = (z_val - h[low_idx,i,j])/(h[high_idx,i,j] - h[low_idx,i,j])
                weight_high = 1. - weight_low

                # Compute and store value
                field_vals[i,j] = weight_low*field_vals_all[low_idx, i, j] + weight_high*field_vals_all[high_idx, i, j]
    return field_vals

def cubic_z_interp(h, field_vals_all, lon, lat, z_val):
    # Now make cubic interpolation coefficients for each grid staggering

    # Use the bottom four levels (in Python notation!)
    levels = [-1, -2, -3, -4]
    
    coeffs = np.zeros((4, len(lat), len(lon)))

    # Compute weights using interpolating polynomials
    coeffs[0] = (
        (z_val - h[levels[1]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[0]] - h[levels[1]]) * (h[levels[0]] - h[levels[2]])
        * (h[levels[0]] - h[levels[3]])
    )
    coeffs[1] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[1]] - h[levels[0]]) * (h[levels[1]] - h[levels[2]])
        * (h[levels[1]] - h[levels[3]])
    )
    coeffs[2] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[2]] - h[levels[0]]) * (h[levels[2]] - h[levels[1]])
        * (h[levels[2]] - h[levels[3]])
    )
    coeffs[3] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[2]])
    ) / (
        (h[levels[3]] - h[levels[0]]) * (h[levels[3]] - h[levels[1]])
        * (h[levels[3]] - h[levels[2]])
    )

    field_vals = np.zeros((len(lat), len(lon)))
    
    for i in np.arange(4):
        field_vals += coeffs[i]*field_vals_all[levels[i]]

    # Set values below surface to NaN
    field_vals = np.where(
        h[levels[0]] > z_val, np.nan, field_vals
    )
    
    return field_vals

In [32]:
# Choose the data to regrid
test = 'gap'
#test='vortex'

rot = False

dycore = 'CAM-SE'

In [33]:
if rot:
    rot_state='with_rot'
else:
    rot_state='omega0'


case = f'cam_6_4_100_se_ne60_ztop20km_L57_new_RF'
if test == 'gap':
    nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau_100s.nc'
else:
    nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau100s.nc'


run_base = "/glade/derecho/scratch/timand/"
run_path = run_base + case + '/run/' + nc_file
nc = Dataset(run_path)

In [34]:
# Save the regridded data. Do so as net cdf
savename = f'{test}_{rot_state}_surface_values'
output_file = Dataset(f'/glade/u/home/timand/dcmip2025_gap_and_vortex/interpolate_data/interp_data/{savename}.nc', 'w')

time = nc['time'][:]
lat = nc['lat'][:] 
lon = nc['lon'][:]

output_file.createDimension('lon', len(lon))
output_file.createDimension('lat', len(lat))

lon_var = output_file.createVariable('lon', 'f4', ('lon',))
lat_var = output_file.createVariable('lat', 'f4', ('lat',))

output_file.variables['lon'][:] = lon
output_file.variables['lat'][:] = lat

ps_var = output_file.createVariable('PS', 'f4', ('lat', 'lon'))
phis_var = output_file.createVariable('PHIS', 'f4', ('lat', 'lon'))

PS = nc['PS'][0, :, :]
PHIS = nc['PHIS'][0, :, :]

output_file.variables['PS'][:, :] = PS
output_file.variables['PHIS'][:, :] = PHIS

output_file.close()